# Triangle communities and the 3-core: predictions, measurements, and conclusions

The same limiting degree law can hide different patterns of neighbour support.
Our triangle graph groups edges into overlapping communities. The configuration
model (CM) with $K=2D$, $D\sim\operatorname{Poisson}(\lambda)$, pairs half-edges
freely. Both have limiting mean degree $2\lambda$ and variance $4\lambda$.

Notebook 05 derives the CM prediction; notebook 08 derives the triangle cavity
prediction. Here we test both against saved measurements and complete the
comparison. We first examine the triangle fixed-point branches, then validate
its density across sizes, inspect individual graphs near onset, and finally
compare the two models on the same experimental grid.

The CM prediction follows an established theorem under its crossing conditions.
The triangle prediction is a **cavity heuristic** based on independent branches
of the limiting bipartite tree, while retaining dependence within each triangle.
Agreement with finite graphs tests that heuristic; it does not prove that the
infinite-size and infinite-pruning-time limits can be interchanged.

This notebook loads definitions and CSVs only. Running it never generates graphs.

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from matplotlib.figure import Figure

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/05_configuration_theory.ipynb"
%run "{NOTEBOOK_DIR}/08_triangle_cavity.ipynb"
%matplotlib inline

PROJECT_ROOT = NOTEBOOK_DIR.parent

## Select saved experiments

`comparison_dir` contains the broad matched-condition study from notebook 06.
`refinement_dir` contains the additional triangle trials from notebook 09. Keep
these designs separate: they use different grids and repeat counts. Raw trials
are never overwritten by analysis. Setting `refinement_dir=None` explicitly
selects a broad-study-only view; missing files in a selected directory raise an
error rather than starting or silently skipping simulations.

The fixed band `critical_exclusion` is omitted only from descriptive error
scores, around each model's own critical parameter. All points remain visible
in the figures and summary CSV. It is an analysis choice, not a rigorous bound
on the finite-size critical window.

`large_core_cutoff` defines a descriptive substantial-core event. Its frequency
crossing is checked against the additional `onset_cutoffs`. These finite-grid
markers help describe finite-size behaviour; they are not threshold estimates
with confidence guarantees.

In [ ]:
config = {
    "comparison_dir": "results/runs/configuration_comparison",  # Reuse notebook 06.
    "refinement_dir": "results/runs/triangle_cavity",  # New triangle trials from 09.
    "figure_dir": "results/figures/triangle_cavity",
    "focus_half_width": 0.065,  # Lambda distance around the triangle prediction.
    "critical_exclusion": 0.025,  # Band omitted from aggregate error scores only.
    "histogram_n": None,  # Use the largest available refinement size.
    "histogram_lambdas": [
        1.22,
        1.2286,
        1.2375,
    ],  # Actual saved values, not interpolations.
    "large_core_cutoff": 0.05,  # At least 5% of all vertices survive.
    "onset_cutoffs": [0.02, 0.05, 0.08],  # Check how the operational bracket changes.
}
config

## Load measurements and retain their provenance

A trial is identified by its source experiment, model, size, parameter, and
replicate. We check for duplicate keys and inconsistent core fractions. Keeping
sources separate avoids giving new measurements extra weight without noticing.
The two CM conventions share a pairing; all repetitions within a model are
independent. CM erasure remains a control, with the convention checks in notebook 07.

For every group we retain the mean, median, quartiles, standard error of the
mean, empty-core frequency, and substantial-core frequency. Quartiles describe
**variation between graphs**; the standard error describes **uncertainty in the
estimated mean**. With one trial, that standard error is unavailable.

In [ ]:
MODEL_LABELS = {
    "triangle_rig": "Triangle RIG",
    "cm_multigraph": "CM multigraph",
    "cm_erased": "CM erased",
}
MODEL_COLORS = {"triangle_rig": "C0", "cm_multigraph": "C1", "cm_erased": "C2"}


def load_triangle_study(comparison_dir, refinement_dir=None) -> list[dict]:
    """Read core measurements from existing studies without loading graph code."""
    sources = [("comparison", Path(comparison_dir), "comparison_n*.csv")]
    if refinement_dir is not None:
        sources.append(("refinement", Path(refinement_dir), "cavity_refinement_n*.csv"))
    rows, seen = [], set()
    for dataset, directory, pattern in sources:
        paths = sorted(directory.glob(pattern))
        if not paths:
            raise FileNotFoundError(
                f"No {pattern} files in {directory}. Run the corresponding simulation notebook first."
            )
        for path in paths:
            with path.open(newline="", encoding="utf-8") as stream:
                for saved in csv.DictReader(stream):
                    row = {"dataset": dataset, "model": saved["model"]}
                    for key in ("n", "replicate", "seed", "core_size"):
                        row[key] = int(saved[key])
                    for key in ("lambda", "core_fraction"):
                        row[key] = float(saved[key])
                    row["trial_seed"] = (
                        int(saved["trial_seed"])
                        if saved.get("trial_seed")
                        else row["seed"]
                    )
                    key = (
                        dataset,
                        row["model"],
                        row["n"],
                        row["lambda"],
                        row["replicate"],
                    )
                    if key in seen or row["model"] not in MODEL_LABELS:
                        raise ValueError(
                            f"Duplicate trial or unknown model in {path}: {key}"
                        )
                    seen.add(key)
                    if dataset == "refinement" and row["model"] != "triangle_rig":
                        raise ValueError("The refinement is a triangle-only experiment")
                    if (
                        row["n"] <= 0
                        or not 0 <= row["core_size"] <= row["n"]
                        or not np.isclose(
                            row["core_fraction"],
                            row["core_size"] / row["n"],
                            atol=1e-12,
                        )
                        or not np.isfinite(row["lambda"])
                        or row["lambda"] < 0
                    ):
                        raise ValueError(f"Invalid measurement in {path}: {key}")
                    # Existing broad diagnostics support the degree/clustering check.
                    for field in (
                        "degree_mean",
                        "degree_variance",
                        "odd_degree_fraction",
                        "simple_transitivity",
                    ):
                        row[field] = float(saved[field]) if saved.get(field) else None
                    rows.append(row)
    return rows


def sample_statistics(values) -> dict:
    """Separate graph-to-graph spread from uncertainty in the simulated mean."""
    values = np.asarray(values, dtype=float)
    return {
        "trials": len(values),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "q25": float(np.percentile(values, 25)),
        "q75": float(np.percentile(values, 75)),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "standard_error": float(np.std(values, ddof=1) / np.sqrt(len(values)))
        if len(values) > 1
        else None,
    }


def summarize_triangle_study(rows, cutoff=0.05) -> list[dict]:
    """Summarize each experimental design independently on its own lambda grid."""
    if not rows or not 0 < cutoff <= 1:
        raise ValueError("Provide measurements and a cutoff in (0, 1]")
    groups = defaultdict(list)
    for row in rows:
        groups[(row["dataset"], row["model"], row["n"], row["lambda"])].append(
            row["core_fraction"]
        )
    return [
        {
            "dataset": dataset,
            "model": model,
            "n": n,
            "lambda": lam,
            **sample_statistics(values),
            "empty_probability": float(np.mean(np.array(values) == 0)),
            "large_core_probability": float(np.mean(np.array(values) >= cutoff)),
        }
        for (dataset, model, n, lam), values in sorted(groups.items())
    ]


def save_table(rows, path):
    """Write a derived table, leaving the source trial CSVs unchanged."""
    if not rows:
        print(f"No rows for {path.name}; table not written.")
        return
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


refinement_dir = (
    PROJECT_ROOT / config["refinement_dir"]
    if config["refinement_dir"] is not None
    else None
)
rows = load_triangle_study(PROJECT_ROOT / config["comparison_dir"], refinement_dir)
summary = summarize_triangle_study(rows, config["large_core_cutoff"])
broad = [row for row in summary if row["dataset"] == "comparison"]
refined = [row for row in summary if row["dataset"] == "refinement"]
if not refined:
    print(
        "Refinement explicitly disabled: the close-up uses the broad triangle study only."
    )
    refined = [row for row in broad if row["model"] == "triangle_rig"]
focus_dataset = refined[0]["dataset"]
broad_sizes = sorted({row["n"] for row in broad})
refined_sizes = sorted({row["n"] for row in refined})
triangle_critical = triangle_critical_point()
cm_critical = cm_critical_point()
figure_dir = PROJECT_ROOT / config["figure_dir"]
figure_dir.mkdir(parents=True, exist_ok=True)
figures = {}
print(
    f"Loaded {len(rows):,} observations; broad sizes {broad_sizes}, focused sizes {refined_sizes}."
)

## What the equations predict

The triangle calculation keeps two message probabilities: $q_1$ and $q_2$ are
the chances that a community supplies one or two surviving neighbours when its
receiving vertex is held present. External support is compound Poisson,
$S=A+2B$, with independent $A\sim\operatorname{Poisson}(\lambda q_1)$ and
$B\sim\operatorname{Poisson}(\lambda q_2)$. This keeps the two children in a
triangle coupled, while separating messages from different communities.

In CM, each incoming edge is a separate branch. One scalar support probability
is enough, but the degree reached along an edge must be size-biased. The two
models therefore share a degree law without sharing their support equations.

The following table compares the positive branch's **right-hand limit** at each
threshold. Neither entry asserts a deterministic finite-size density exactly at
the critical parameter. The triangle values characterize its cavity equations;
the CM values also have the theorem-based interpretation explained in notebook 05.

In [ ]:
critical_comparison = [
    {
        "model": "triangle_rig",
        "status": "cavity heuristic",
        "lambda_c": triangle_critical["lambda"],
        "mean_degree_c": 2 * triangle_critical["lambda"],
        "predicted_jump": triangle_critical["rho"],
        "transition": "discontinuous",
        "fixed_point_residual": triangle_critical["residual"],
        "tangent_residual": triangle_critical["tangent_residual"],
    },
    {
        "model": "cm_multigraph",
        "status": "CM theorem away from tangency",
        "lambda_c": cm_critical["lambda"],
        "mean_degree_c": 2 * cm_critical["lambda"],
        "predicted_jump": cm_critical["rho"],
        "transition": "discontinuous",
        "fixed_point_residual": cm_critical["residual"],
        "tangent_residual": cm_critical["tangent_residual"],
    },
]
display(
    Markdown(
        "| Model | Prediction status | Critical lambda | Positive density limit | Transition |\n"
        "|---|---|---:|---:|---|\n"
        + "\n".join(
            f"| {MODEL_LABELS[row['model']]} | {row['status']} | {row['lambda_c']:.6f} | {row['predicted_jump']:.6f} | {row['transition']} |"
            for row in critical_comparison
        )
    )
)
print(
    f"Triangle threshold reduction relative to CM: {100 * (1 - triangle_critical['lambda'] / cm_critical['lambda']):.2f}%."
)

## Inspect the positive fixed-point branches

Notebook 08 parameterizes every positive fixed point by $z=-\log p_0$, where
$p_0$ is the probability of no external support. Larger $z$ means more support.
The minimum of $\Lambda(z)$ identifies the first positive solution. The branch
with larger $z$ is selected by pruning from all messages active; the other
positive branch is not reached by that initialization.

The nonzero density at the fold establishes a discontinuity **in this cavity
prediction**. The zero fixed point persists on both sides. The initial parameter
grid brackets the fold; continuous bisection, rather than the grid spacing,
sets the reported numerical location.

In [ ]:
branch_points = [triangle_branch(z) for z in np.linspace(0.03, 3.0, 700)]
figure = Figure(figsize=(10, 4.2), layout="constrained")
left, right = figure.subplots(1, 2)
for selected, style, label in [
    (
        [p for p in branch_points if p["z"] < triangle_critical["z"]],
        "--",
        "Other positive branch",
    ),
    (
        [p for p in branch_points if p["z"] >= triangle_critical["z"]],
        "-",
        "Maximal branch",
    ),
]:
    left.plot(
        [p["z"] for p in selected],
        [p["lambda"] for p in selected],
        style,
        color="C0",
        label=label,
    )
    right.plot(
        [p["lambda"] for p in selected],
        [p["rho"] for p in selected],
        style,
        color="C0",
        label=label,
    )
left.scatter(
    triangle_critical["z"], triangle_critical["lambda"], color="black", zorder=4
)
left.axhline(triangle_critical["lambda"], color="black", ls=":", lw=1)
left.set(
    xlabel=r"$z=-\log p_0$",
    ylabel=r"$\Lambda(z)$",
    ylim=(1.15, 2.2),
    title="A positive pair of solutions appears at the fold",
)
right.plot([1.15, 1.5], [0, 0], color="gray", ls=":", label="Zero fixed point")
right.plot(
    triangle_critical["lambda"], triangle_critical["rho"], "o", mfc="white", mec="black"
)
right.set(
    xlabel=r"$\lambda$",
    ylabel="Predicted 3-core fraction",
    xlim=(1.15, 1.5),
    ylim=(-0.01, 0.4),
    title="Pruning selects a positive density at onset",
)
for axis in (left, right):
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
figures["branches"] = figure
display(figure)

## Check numerical convergence independently of the scalar solver

The branch solver avoids long iterations near a fold. As an independent check,
we also apply the original two-dimensional probability update, starting with
every message equal to two. The tails $q_1+q_2$ and $q_2$ decrease; $q_1$ itself
need not decrease, since a two-neighbour message can become a one-neighbour
message before disappearing.

The table checks several stopping tolerances just below and above the fold.
The trace at exactly the critical parameter illustrates slow convergence; a
finite iteration there is not used to define the critical density. A small
update residual near a unit Jacobian eigenvalue can coexist with a larger
solution error, so the table records both residual and error against the scalar
solution. These numerical checks do not establish the large-graph heuristic.

In [ ]:
# Independent critical calculation: minimize the explicit branch directly,
# rather than solving its analytic derivative as in notebook 08.
lower, upper = 0.1, 2.0
for _ in range(70):
    first = lower + (upper - lower) / 3
    second = upper - (upper - lower) / 3
    if triangle_branch(first)["lambda"] < triangle_branch(second)["lambda"]:
        upper = second
    else:
        lower = first
minimum_check = triangle_branch((lower + upper) / 2)
minimum_error = abs(minimum_check["lambda"] - triangle_critical["lambda"])
q_critical = [triangle_critical["q1"], triangle_critical["q2"]]
eigenvalues = np.linalg.eigvals(
    triangle_jacobian(q_critical, triangle_critical["lambda"])
)
unit_eigenvalue_error = float(np.min(abs(eigenvalues - 1)))
if minimum_error > 1e-10 or unit_eigenvalue_error > 1e-9:
    raise RuntimeError("Independent critical-point checks disagree")
critical_comparison[0]["independent_lambda_error"] = minimum_error
critical_comparison[0]["unit_eigenvalue_error"] = unit_eigenvalue_error
critical_comparison[1]["independent_lambda_error"] = None
critical_comparison[1]["unit_eigenvalue_error"] = None
print(
    f"Independent minimization: lambda error={minimum_error:.2e}; "
    f"unit-eigenvalue error={unit_eigenvalue_error:.2e}."
)

numerical_diagnostics = []
for lam in [
    triangle_critical["lambda"] - 0.001,
    triangle_critical["lambda"] + 0.001,
    1.5,
]:
    prediction = triangle_prediction(lam, critical=triangle_critical)
    for tolerance in [1e-9, 1e-12, 1e-14]:
        iteration = triangle_iteration(lam, tolerance=tolerance)
        numerical_diagnostics.append(
            {
                "lambda": lam,
                "tolerance": tolerance,
                "iterations": iteration["iterations"],
                "converged": iteration["converged"],
                "iteration_residual": iteration["residual"],
                "density_error": abs(iteration["rho"] - prediction["rho"]),
                "scalar_residual": prediction["residual"],
            }
        )
        if not iteration["converged"]:
            raise RuntimeError(
                "The independent iteration did not converge; inspect numerical_diagnostics"
            )
figure = Figure(figsize=(8, 4.2), layout="constrained")
axis = figure.subplots()
for offset, color in [(-0.001, "C2"), (0.0, "black"), (0.001, "C0")]:
    lam = triangle_critical["lambda"] + offset
    q = np.array([0.0, 1.0])
    trace = []
    for step in range(1200):
        q = triangle_messages(q, lam)
        trace.append(triangle_density(q, lam))
    axis.plot(
        np.arange(1, len(trace) + 1), trace, color=color, label=rf"$\lambda={lam:.6f}$"
    )
axis.set(
    xlabel="Message-update round",
    ylabel="Core-density estimate after this update",
    xscale="log",
    title="A long pruning bottleneck near the critical point",
)
axis.grid(alpha=0.25)
axis.legend()
figures["iteration"] = figure
display(figure)
for row in numerical_diagnostics:
    print(
        f"lambda={row['lambda']:.6f}, tolerance={row['tolerance']:.0e}: "
        f"{row['iterations']} rounds; density error={row['density_error']:.2e}"
    )

## Validate the triangle density on the saved broad study

Solid curves are predictions; markers and thin dashed connections are simulated
means. Shading contains the middle half of individual graphs, while bars show
one standard error of the mean. Connections between sampled means are guides;
there is no smoothing or fitted transition curve.

We draw zero and positive prediction branches separately, with an open circle
at the positive critical limit. Around a discontinuous transition, some finite
graphs can have empty cores and others substantial ones. Their average can lie
between typical outcomes and need not follow the limiting curve there.

In [ ]:
all_lambdas = [row["lambda"] for row in rows]
theory_grid = sorted(
    set(np.linspace(min(all_lambdas), max(all_lambdas), 650))
    | set(all_lambdas)
    | {triangle_critical["lambda"], cm_critical["lambda"]}
)
theory_curves = []
for model, critical, predictor in [
    ("triangle_rig", triangle_critical, triangle_prediction),
    ("cm_multigraph", cm_critical, cm_prediction),
]:
    for lam in theory_grid:
        result = predictor(lam, critical=critical)
        theory_curves.append(
            {
                "model": model,
                "lambda": lam,
                "rho": result["rho"],
                "residual": result["residual"],
            }
        )


def draw_prediction(axis, model):
    """Keep discontinuous branches separate and label the positive branch limit."""
    critical = triangle_critical if model == "triangle_rig" else cm_critical
    selected = [row for row in theory_curves if row["model"] == model]
    for above in [False, True]:
        branch = [
            row for row in selected if (row["lambda"] >= critical["lambda"]) == above
        ]
        axis.plot(
            [row["lambda"] for row in branch],
            [row["rho"] for row in branch],
            color=MODEL_COLORS[model],
            lw=2,
            label=("Triangle cavity" if model == "triangle_rig" else "CM prediction")
            if above
            else None,
        )
    axis.axvline(critical["lambda"], color=MODEL_COLORS[model], ls=":", alpha=0.7, lw=1)
    axis.plot(
        critical["lambda"],
        critical["rho"],
        "o",
        mfc="white",
        mec=MODEL_COLORS[model],
        ms=6,
    )


def draw_sample(axis, selected, color, label):
    """Mean, middle half of graphs, and one standard error of the mean."""
    selected = sorted(selected, key=lambda row: row["lambda"])
    grid = [row["lambda"] for row in selected]
    axis.fill_between(
        grid,
        [row["q25"] for row in selected],
        [row["q75"] for row in selected],
        color=color,
        alpha=0.12,
    )
    axis.plot(
        grid,
        [row["mean"] for row in selected],
        "o--",
        ms=3,
        lw=0.9,
        color=color,
        label=label,
    )
    available = [row for row in selected if row["standard_error"] is not None]
    axis.errorbar(
        [row["lambda"] for row in available],
        [row["mean"] for row in available],
        yerr=[row["standard_error"] for row in available],
        fmt="none",
        ecolor=color,
        capsize=2,
        lw=0.8,
    )


def core_axes(axis):
    axis.set(xlabel=r"$\lambda$", ylabel="3-core fraction")
    axis.grid(alpha=0.25)


figure = Figure(figsize=(9, 5.3), layout="constrained")
axis = figure.subplots()
for index, n in enumerate(broad_sizes):
    selected = [
        row for row in broad if row["model"] == "triangle_rig" and row["n"] == n
    ]
    draw_sample(axis, selected, f"C{index + 2}", f"Triangle mean; n={n:,}")
draw_prediction(axis, "triangle_rig")
core_axes(axis)
axis.set(
    ylim=(-0.01, 1),
    title="Triangle cavity prediction versus the saved broad study\nShading: middle half of graphs; bars: one standard error of mean",
)
axis.legend()
figures["triangle_validation"] = figure
display(figure)

## Resolve the finite-size rise with the focused trials

Each panel shows one size on the same lambda and density scales. The predicted
jump stays fixed while the distribution of finite-graph outcomes changes.
The additional largest size helps distinguish a persistent disagreement from a
finite-size rounding effect. Individual trial distributions are examined next.

In [ ]:
columns = min(2, len(refined_sizes))
row_count = (len(refined_sizes) + columns - 1) // columns
figure = Figure(figsize=(5.2 * columns, 4 * row_count), layout="constrained")
axes = figure.subplots(row_count, columns, squeeze=False).ravel()
for axis, n in zip(axes, refined_sizes):
    selected = [row for row in refined if row["n"] == n]
    draw_sample(axis, selected, "black", "Simulated triangle mean")
    draw_prediction(axis, "triangle_rig")
    core_axes(axis)
    axis.set(
        xlim=(
            triangle_critical["lambda"] - config["focus_half_width"],
            triangle_critical["lambda"] + config["focus_half_width"],
        ),
        ylim=(-0.005, 0.25),
        title=f"n={n:,}",
    )
    axis.legend(fontsize=8)
for axis in axes[len(refined_sizes) :]:
    axis.set_visible(False)
figure.suptitle(
    f"Triangle transition; {focus_dataset} data\nShading: middle half of graphs; bars: one standard error of mean"
)
figures["refinement"] = figure
display(figure)

validation_errors = []
for group in summary:
    if group["model"] != "cm_erased":
        predictor, critical = (
            (triangle_prediction, triangle_critical)
            if group["model"] == "triangle_rig"
            else (cm_prediction, cm_critical)
        )
        group["prediction"] = predictor(group["lambda"], critical=critical)["rho"]
        group["bias"] = group["mean"] - group["prediction"]
    else:
        group["prediction"] = None
        group["bias"] = None
for dataset, model, n in sorted(
    {(r["dataset"], r["model"], r["n"]) for r in summary if r["model"] != "cm_erased"}
):
    critical = triangle_critical if model == "triangle_rig" else cm_critical
    selected = [
        r
        for r in summary
        if (r["dataset"], r["model"], r["n"]) == (dataset, model, n)
        and abs(r["lambda"] - critical["lambda"]) > config["critical_exclusion"]
    ]
    if selected:
        errors = np.array([r["bias"] for r in selected])
        validation_errors.append(
            {
                "dataset": dataset,
                "model": model,
                "n": n,
                "lambda_points": len(selected),
                "excluded_half_width": config["critical_exclusion"],
                "rmse": float(np.sqrt(np.mean(errors**2))),
                "mean_absolute_error": float(np.mean(abs(errors))),
                "maximum_absolute_error": float(np.max(abs(errors))),
            }
        )
for result in validation_errors:
    print(
        f"{result['dataset']:10s} {result['model']:14s} n={result['n']:6,d}: "
        f"RMSE={result['rmse']:.5f} on {result['lambda_points']} points"
    )

## Inspect the individual outcomes near the predicted threshold

Common bins show the fraction of graphs in each interval, with a finer density
range than the broad-study plot. The vertical line marks the cavity density
when it is positive; below the fold the prediction is zero. A peak near zero
and another at positive size explains how averaging can hide an abrupt rise.
The first bin can include small nonempty cores, so the summary table separately
records exact emptiness.

The value $1.2286$ is slightly above the computed critical parameter. It is
labelled with its sampled value; treating a rounded parameter as exactly
critical would give a misleading test of the limiting equations.

In [ ]:
histogram_n = (
    refined_sizes[-1] if config["histogram_n"] is None else config["histogram_n"]
)
if histogram_n not in refined_sizes:
    raise ValueError("histogram_n must select a saved focused-study size")
available = {row["lambda"] for row in refined if row["n"] == histogram_n}
histogram_lambdas = config["histogram_lambdas"]
if not histogram_lambdas or any(lam not in available for lam in histogram_lambdas):
    raise ValueError("histogram_lambdas must select actual saved focused-study values")
figure = Figure(figsize=(4.7 * len(histogram_lambdas), 4), layout="constrained")
axes = figure.subplots(1, len(histogram_lambdas), squeeze=False)[0]
for axis, lam in zip(axes, histogram_lambdas):
    values = [
        row["core_fraction"]
        for row in rows
        if (row["dataset"], row["model"], row["n"], row["lambda"])
        == (focus_dataset, "triangle_rig", histogram_n, lam)
    ]
    axis.hist(
        values,
        bins=np.linspace(0, 1, 101),
        weights=np.ones(len(values)) / len(values),
        color="C0",
        alpha=0.65,
        label=f"{len(values)} independent graphs",
    )
    prediction = triangle_prediction(lam, critical=triangle_critical)["rho"]
    axis.axvline(
        prediction, color="black", ls="--", label=f"Cavity density {prediction:.3f}"
    )
    axis.set(
        xlabel="3-core fraction",
        ylabel="Fraction of trials in bin",
        title=rf"$\lambda={lam:g}$",
        xlim=(0, max(0.25, max(values) + 0.02)),
        ylim=(0, 1.04),
    )
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)
figure.suptitle(
    f"Triangle outcomes near onset; n={histogram_n:,}; {focus_dataset} data"
)
figures["distributions"] = figure
display(figure)

## Track a descriptive onset event and its sensitivity

We plot the fraction of trials with a core occupying at least the chosen
positive cutoff. Bars are approximate 95% **Wilson intervals for that event
probability**, distinct from the core-fraction quartiles and mean standard errors
above. The prediction's critical parameter is shown as a reference, not fitted
to these probabilities.

The table brackets the first sampled lambda where this event occurs in at least
half the trials. No interpolation or smoothing is used; fluctuations can make
the empirical frequencies nonmonotone. The table also varies the cutoff.
A cutoff above a model's jump would move its limiting event crossing above its
critical point. Even a cutoff below the jump has finite-size and grid effects.
Missing lower or upper endpoints indicate incomplete coverage, not a numerical
critical estimate.

In [ ]:
def wilson_interval(successes, trials):
    """Approximate 95% interval for an independently sampled binary event."""
    z = 1.959963984540054
    p = successes / trials
    denominator = 1 + z**2 / trials
    centre = (p + z**2 / (2 * trials)) / denominator
    width = z * np.sqrt(p * (1 - p) / trials + z**2 / (4 * trials**2)) / denominator
    return centre - width, centre + width


onset_brackets = []
for cutoff in config["onset_cutoffs"]:
    cutoff_summary = summarize_triangle_study(rows, cutoff)
    for dataset, model, n in sorted(
        {(r["dataset"], r["model"], r["n"]) for r in cutoff_summary}
    ):
        selected = [
            r
            for r in cutoff_summary
            if (r["dataset"], r["model"], r["n"]) == (dataset, model, n)
        ]
        selected.sort(key=lambda r: r["lambda"])
        crossing = next(
            (i for i, r in enumerate(selected) if r["large_core_probability"] >= 0.5),
            None,
        )
        onset_brackets.append(
            {
                "dataset": dataset,
                "model": model,
                "n": n,
                "core_fraction_cutoff": cutoff,
                "lambda_lower": selected[crossing - 1]["lambda"]
                if crossing is not None and crossing > 0
                else None,
                "lambda_upper": selected[crossing]["lambda"]
                if crossing is not None
                else None,
            }
        )
figure = Figure(figsize=(8.5, 4.5), layout="constrained")
axis = figure.subplots()
for index, n in enumerate(refined_sizes):
    selected = sorted([r for r in refined if r["n"] == n], key=lambda r: r["lambda"])
    probabilities = np.array([r["large_core_probability"] for r in selected])
    intervals = np.array(
        [
            wilson_interval(
                round(r["large_core_probability"] * r["trials"]), r["trials"]
            )
            for r in selected
        ]
    )
    axis.errorbar(
        [r["lambda"] for r in selected],
        probabilities,
        yerr=np.maximum(
            0,
            np.array(
                [probabilities - intervals[:, 0], intervals[:, 1] - probabilities]
            ),
        ),
        marker="o",
        ms=3,
        capsize=2,
        lw=1,
        color=f"C{index}",
        label=f"n={n:,}",
    )
axis.axvline(
    triangle_critical["lambda"], color="black", ls=":", label="Cavity critical lambda"
)
axis.axhline(0.5, color="gray", lw=0.8, ls="--")
axis.set(
    xlabel=r"$\lambda$",
    ylabel="Probability of exceeding the core cutoff",
    ylim=(-0.03, 1.03),
    title=f"Core fraction at least {config['large_core_cutoff']:.0%}; {focus_dataset} data\nBars: approximate 95% Wilson intervals for the event probability",
)
axis.grid(alpha=0.25)
axis.legend(fontsize=8)
figures["onset"] = figure
display(figure)
for r in onset_brackets:
    if r["model"] == "triangle_rig" and r["dataset"] == focus_dataset:
        print(
            f"n={r['n']:6,d}, cutoff={r['core_fraction_cutoff']:.2f}: ({r['lambda_lower']}, {r['lambda_upper']}]"
        )

## Compare both models under the same experimental conditions

These panels use only notebook 06's matched sizes, lambda grid, and 60 repeats
in the default run. Each model receives its own prediction. The CM erased
measurements are shown as a simple-projection control; they share the CM
pairings and do not constitute additional independent CM repetitions.

The comparison concerns the same normalized quantity, $|\operatorname{Core}_3|/n$,
and the same membership parameter $\lambda$. The earlier triangle onset is not
explained by replacing $2\operatorname{Poisson}(\lambda)$ with a Poisson degree
law of the same mean. Notebook 07 checks full degree distributions and finite-size
projection effects; its degree and transitivity diagnostics are reused below.

In [ ]:
figure = Figure(figsize=(5 * len(broad_sizes), 4.8), layout="constrained")
axes = figure.subplots(1, len(broad_sizes), squeeze=False)[0]
for axis, n in zip(axes, broad_sizes):
    for model in MODEL_LABELS:
        selected = [r for r in broad if r["n"] == n and r["model"] == model]
        draw_sample(axis, selected, MODEL_COLORS[model], MODEL_LABELS[model] + " mean")
    draw_prediction(axis, "triangle_rig")
    draw_prediction(axis, "cm_multigraph")
    core_axes(axis)
    axis.set(
        xlim=(triangle_critical["lambda"] - config["focus_half_width"], 1.55),
        ylim=(-0.005, 0.4),
        title=f"n={n:,}",
    )
    axis.legend(fontsize=7)
figure.suptitle(
    "Same limiting degree law, different patterns of neighbour support\nSolid: predictions; markers: means; shading: graph quartiles; bars: mean standard errors"
)
figures["models"] = figure
display(figure)

# Pair differences within the original experimental trial, using the shared trial key.
paired = defaultdict(dict)
for row in rows:
    if row["dataset"] == "comparison":
        paired[(row["n"], row["lambda"], row["replicate"])][row["model"]] = row
samples = defaultdict(list)
for (n, lam, replicate), trial in paired.items():
    if (
        set(trial) != set(MODEL_LABELS)
        or len({r["trial_seed"] for r in trial.values()}) != 1
    ):
        raise ValueError(
            "Each broad trial must contain all three model measurements from one trial seed"
        )
    samples[(n, lam)].append(
        trial["triangle_rig"]["core_fraction"] - trial["cm_multigraph"]["core_fraction"]
    )
model_differences = [
    {"n": n, "lambda": lam, **sample_statistics(values)}
    for (n, lam), values in sorted(samples.items())
]
figure = Figure(figsize=(9, 4.7), layout="constrained")
axis = figure.subplots()
for index, n in enumerate(broad_sizes):
    selected = [r for r in model_differences if r["n"] == n]
    axis.errorbar(
        [r["lambda"] for r in selected],
        [r["mean"] for r in selected],
        yerr=[r["standard_error"] or 0 for r in selected],
        marker="o",
        ms=3,
        lw=1,
        capsize=2,
        color=f"C{index + 2}",
        label=f"Measured difference; n={n:,}",
    )
# Split at both folds: joining across either jump would invent intermediate predictions.
for lower, upper in [
    (min(theory_grid), triangle_critical["lambda"]),
    (triangle_critical["lambda"], cm_critical["lambda"]),
    (cm_critical["lambda"], max(theory_grid) + 1),
]:
    grid = [lam for lam in theory_grid if lower <= lam < upper]
    difference = [
        triangle_prediction(lam, critical=triangle_critical)["rho"]
        - cm_prediction(lam, critical=cm_critical)["rho"]
        for lam in grid
    ]
    axis.plot(
        grid,
        difference,
        color="black",
        lw=2,
        label="Predicted triangle minus CM" if lower == min(theory_grid) else None,
    )
axis.axhline(0, color="gray", lw=0.8)
axis.set(
    xlabel=r"$\lambda$",
    ylabel="Triangle minus CM multigraph: core fraction",
    title="Earlier emergence does not imply a larger core at every lambda\nBars: one standard error of the within-trial mean difference",
)
axis.grid(alpha=0.25)
axis.legend(fontsize=8)
figures["model_difference"] = figure
display(figure)

## Save the scientific outputs

The critical table records prediction status as well as numerical values. Error
summaries are grouped by experiment and size; the broad and focused grids are
not interchangeable convergence metrics. Near the transition, inspect the
curves and distributions rather than relying on an aggregate error score.

The theory CSV records the residuals, so numerical accuracy can be distinguished
from disagreement with finite graphs. Generated outputs remain under `results/`
and are ignored by Git. Selected figures can later be copied into `report/figures/`.

In [ ]:
for name, table in [
    ("summary", summary),
    ("validation_errors", validation_errors),
    ("critical_comparison", critical_comparison),
    ("theory_curves", theory_curves),
    ("onset_brackets", onset_brackets),
    ("model_differences", model_differences),
    ("numerical_diagnostics", numerical_diagnostics),
]:
    save_table(table, figure_dir / f"{name}.csv")
for name, figure in figures.items():
    figure.savefig(figure_dir / f"{name}.png", dpi=180)
print(f"Saved {len(figures)} figures and seven analysis tables to {figure_dir}.")

## Findings from the completed reference experiments

Notebook 06's saved study contains 60 trials at each of 18 lambda values for
$n=1{,}000,3{,}000,10{,}000$. We reused all 9,720 measurements. Notebook 09 added
4,000 triangle graphs: 100 independent repetitions at ten focused lambda values
for each of $n=1{,}000,3{,}000,10{,}000,30{,}000$, using seed `20261110`.
These figures were generated and inspected after both studies had finished.
The numbers below describe this reference run; changed configurations should
be interpreted using their own output tables.

**Agreement improves with graph size away from the rise.** For the 17 broad-grid
triangle points outside $|\lambda-\lambda_{c,\triangle}|\le0.025$, the RMSE of
simulated means against the cavity curve is $0.0197$, $0.0132$, and $0.00307$
as size increases. On the three focused-grid points outside that same band
($1.19,1.255,1.27$), it is $0.0389$, $0.0202$, $0.00782$, and $0.000207$.
These are separate grids and finite Monte Carlo summaries, not convergence
rates or uniform error bounds.

At $n=30{,}000$ and $\lambda=1.255$, the measured core fraction is
$0.13671\pm0.00144$ (mean and one standard error), versus the cavity value
$0.13642$. At $\lambda=1.27$, it is $0.15341\pm0.00117$, versus $0.15361$.
All 100 graphs have nonempty cores at both points. At $\lambda=1.19$, all 100
are empty at this size, compared with 49 nonempty graphs out of 100 at $n=1{,}000$.
This is consistent with the predicted zero branch below the transition and
positive branch above it, with a finite-size window narrowing between them.

**The critical region still has mixed outcomes.** At $\lambda=1.22$ and
$n=30{,}000$, 71 graphs are empty and 29 have substantial cores. At $\lambda=1.2286$,
42 are empty and 58 have substantial cores; the latter have mean density about
$0.10770$. Averaging gives $0.06246$, below the cavity value $0.08148$ at this
slightly supercritical parameter. At $\lambda=1.2375$, 18 graphs are still empty.
The coexistence of zero and positive outcomes supports the discontinuous
picture, while showing why means near onset need not match the asymptotic curve.
There is no evidence here of a numerical branch-selection failure.

**The empirical onset markers move toward the predicted region, with limits.**
For the event “at least 5% survives”, the first sampled 50% crossing lies in
$(1.19,1.21]$, $(1.21,1.22]$, $(1.22,1.225]$, and $(1.22,1.225]$ across the four
sizes. The last bracket remains below the cavity value $1.22860$; it is a
descriptive marker, not a threshold confidence interval. At $n=30{,}000$ the
frequency at $1.225$ is only $54/100$, so binomial uncertainty is material.
Raising the cutoff from $0.05$ to $0.08$ moves this size's bracket to
$(1.225,1.2286]$, illustrating cutoff sensitivity near the predicted $0.08111$ jump.

**Triangle structure advances emergence but does not increase every core.**
The two predicted thresholds differ by about $0.05912$ in lambda, or 4.59%
relative to CM. The triangle jump is about 8.11% of vertices, compared with
13.79% for CM. At $n=10{,}000$ and $\lambda=1.26$, the saved triangle and CM means
are $0.14106$ and $0.00980$, respectively. Their ordering reverses farther above
onset: at $\lambda=1.5$, the predictions are about $0.32538$ and $0.34100$, and
the measured means are $0.32456$ and $0.34113$. The measured triangle-minus-CM
difference is $-0.01657\pm0.00209$, using the standard error of within-trial
differences. The predicted curves cross near $\lambda=1.31$.

At that latter parameter $\lambda=1.5$, the saved $n=10{,}000$ degree diagnostics
are close to their common limiting law: triangle/CM means $2.9961/2.9976$ and
variances $5.9912/5.9890$, against targets 3 and 6. Simple-projection transitivity
is about $0.2505$ for triangles and $0.00054$ for CM. The degree match and erasure
checks in notebook 07 help distinguish this community-structure comparison from
a comparison that changes only the degree distribution. They do not isolate
clustering from every other correlation created by triangle communities.


## Discussion: what changes when edges form triangle communities?

**The mathematical prediction.** The cavity equations predict a lower triangle
threshold, about $1.22860$ rather than the CM value $1.28772$, and a smaller jump,
about $0.08111$ rather than $0.13789$. A pair of children with one external
support each can support one another inside a triangle. Accounting for this
joint survival changes the recursion even though the limiting degree law is the
same. Both equations predict discontinuous emergence. Their density curves can
cross above onset, so triangle structure need not increase the core at every
parameter value.

**What finite simulations can show.** Agreement away from the rise and a narrowing
mixture of small and substantial cores near it support the prediction. The
finite-size event brackets and histogram shapes remain empirical evidence.
They do not prove an asymptotic jump, a convergence rate, or an exact finite-size
scaling law. The reference-run measurements above quantify both agreement away from the
transition and the remaining finite-size rounding near it.

**What remains heuristic.** Local weak convergence justifies the independent
branches at every fixed depth of the bipartite graph. It does not by itself
control a pruning process with a number of rounds growing with $n$. Rare repeated
memberships and shared edges are also merged in the finite simple projection;
near criticality even a small change can trigger a deletion cascade. Their
negligible local frequency is not a proof of global core insensitivity.

**How to interpret clustering.** The degree diagnostics in notebook 07 support
the intended limiting match. Triangle transitivity stays substantial, whereas
CM simple-projection transitivity decreases with size. This comparison changes
community organisation and its associated correlations together. It identifies
an effect beyond the marginal degree law, without isolating a causal effect of
one scalar clustering coefficient.

The triangle equations are derived from the assignment's Section 2 in notebook 08.
[van der Hofstad, Komjáthy and Vadon (2021), Theorem 2.14 and Corollary 2.9](https://arxiv.org/pdf/1809.02514)
provide the bipartite local limit and degree law, not a 3-core theorem.
[Janson and Łuczak (2007), Theorem 2.3](https://arxiv.org/pdf/math/0508453)
provides the CM benchmark. [Gleeson and Melnik (2009), Section V](https://arxiv.org/pdf/0811.4511)
illustrates clique-aware pruning for a different ensemble; its formulas are not
substituted for our overlapping-triangle recursion.